In [19]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from aicsimageio import AICSImage
import time
import zarr
import os 
import sys
from pathlib import Path
import shutil
import re
from tifffile import imwrite

### Do not change the code in the cell below

In [ ]:
# This assumes that your notebook is inside the folder 'full movie', which is inside 'movie_data'
base_dir =  r'\\10.158.28.194\ActiveNAS\Kenny\SiRActinData'

# Define the file directory and name
input_file_directory = '100nM_SiRactin_analysis/'
save_file_directory = input_file_directory + 'zarr_file'
save_full_path = os.path.join(base_dir, save_file_directory)

zarr_directory = save_file_directory + '/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_directory)

# Follow the instructions to properly run the notebook 
1. Save the your full movie in tif format to the following directory **LLSM-CME-ANALYSIS/Final/movie_data/full_movie/**
2. Movie must be in tif format 
3. Change the **input_file_name** below to match your movie name so it is loaded properly 
 
 **Nothing else needs to be changed**




In [ ]:
######## Change name of movie file here #######
input_file_name = '20240910_MA2_Abhi410_63_510_202020_15um_noalign_S100nMV10uM_230_3h-01_processed.czi'
print('Your file name is: ', input_file_name)

In [ ]:
# Full path construction
input_full_path = os.path.join(base_dir, input_file_directory, input_file_name)

# Load the file
img = AICSImage(input_full_path)
# Check which reader is being used
print(type(img.reader))

In [ ]:
dask_array = img.xarray_dask_data 
dask_array.name = 'all_channels_data'
# dask_array

In [ ]:
dask_array.attrs = []
dask_array.to_zarr(store = save_full_path, mode = 'w', compute = True)

In [ ]:
z2 = zarr.open(zarr_full_path, mode='r')
z2.info

In [ ]:
# ## NEW CODE FOR VALERIE -- CREATES TIFF FILES FOR MATLAB

# from tifffile import imwrite  # Optional, for saving TIFFs instead of zarrs

# # Output directory
# output_dir = "ch3"
# os.makedirs(output_dir, exist_ok=True)

# # Loop over timepoints and save each one
# for t in range(dask_array.sizes["T"]):
#     # Select timepoint t and channel 3 (adjust index if C is coordinate-labeled)
#     volume = dask_array.isel(T=t, C=1).compute()  # Now a numpy array with shape (Z, Y, X)
    
#     # Convert to desired dtype if needed (e.g., uint16)
#     volume_np = volume.astype(np.uint16)
    
#     # Save to TIFF
#     imwrite(os.path.join(output_dir, f"test_ch3_t{t:03d}.tif"), volume_np)

# print(f"Saved {dask_array.sizes['T']} TIFFs to {output_dir}")

In [46]:
def create_zar(movie, output_dir):
    img = AICSImage(movie)
    dask_array = img.xarray_dask_data 
    dask_array.name = 'all_channels_data'
    dask_array.attrs = []
    dask_array.to_zarr(store = output_dir, mode = 'w', compute = True)

In [ ]:
# def create_matlab_import():
    

In [ ]:
def create_tiff(z_file, channels_to_tiff, output_dir):
    
    z = zarr.open(z_file, mode='r')
    
    time = z.shape[0]
    for c in channels_to_tiff:
        channel_dir = os.path.join(output_dir, f"ch{c+1}")
        os.makedirs(channel_dir, exist_ok=True)
        
        for t in range(time):
            volume = z[t, c, :, :, :]  # Now a numpy array with shape (Z, Y, X)
            volume_np = volume.astype(np.uint16)
            imwrite(os.path.join(channel_dir, f"ch{c+1}_t{t:03d}.tif"), volume_np)
    
    print(f"Saved TIFFs for channels {channels_to_tiff} to {output_dir}")

In [ ]:
def organize_czi_files(
    root_dir,
    keywords,
    destination_dir,
    channels_to_tiff=[0, 1, 2]
):
    """
    Find .czi files containing specified keywords and create folders
    named after each CZI file.

    Parameters
    ----------
    root_dir : str or Path
        Directory to search for .czi files
    keywords : list[str]
        Keywords \of the czi files to be processed
    destination_dir : str or Path
        Where new folders should be created (defaults to root_dir)
    channels_to_tiff : list[int]
        List of channel indices to create TIFF files for (0-based indexing)
    """

    root_dir = Path(root_dir)
    destination_dir = Path(destination_dir) if destination_dir else root_dir

    keywords = [k.lower() for k in keywords]

    for czi_file in root_dir.rglob("*.czi"):
        filename_lower = czi_file.name.lower()
        # Check that movies with keywords are present
        if any(k in filename_lower for k in keywords):
            folder_name = czi_file.stem  # filename without extension

            # create zar folder location
            zar_name = folder_name + "_analysis"
            zar_folder = destination_dir / zar_name
            zar_folder.mkdir(parents=True, exist_ok=True)
            
            # create zar file 
            zar_file_path = zar_folder / 'zarr_file'
            zar_file_path.mkdir(parents=True, exist_ok=True)
            create_zar(czi_file, zar_file_path)

            # Create tiff folder location
            tiff_folder = destination_dir / folder_name
            tiff_folder.mkdir(parents=True, exist_ok=True)
            
            # create tiff files for specifed channels
            zar_file = zar_file_path / 'all_channels_data'
            create_tiff(zar_file, channels_to_tiff, tiff_folder)


            print(f"Processed: {czi_file.name}")

## Testing

In [ ]:
base_dir = r'\\10.158.28.194\ActiveNAS\Kenny\testing\to_do\24h_SlideA1_Channel3'
keywords = ["processed","bleh"]
destination_dir = r'\\10.158.28.194\ActiveNAS\Kenny\testing\processed'
channels_to_tiff=[0, 1, 2]
organize_czi_files(base_dir, keywords, destination_dir, channels_to_tiff)

Saved TIFFs for channels [0, 1, 2] to \\10.158.28.194\ActiveNAS\Kenny\testing\processed\20251118_24h_SlideA1_Channel3-01_processed
Processed: 20251118_24h_SlideA1_Channel3-01_processed.czi
Saved TIFFs for channels [0, 1, 2] to \\10.158.28.194\ActiveNAS\Kenny\testing\processed\20251118_24h_SlideA1_Channel3-02_processed
Processed: 20251118_24h_SlideA1_Channel3-02_processed.czi
Saved TIFFs for channels [0, 1, 2] to \\10.158.28.194\ActiveNAS\Kenny\testing\processed\20251118_24h_SlideA1_Channel3-03_processed
Processed: 20251118_24h_SlideA1_Channel3-03_processed.czi
